# Lab 9 - Bootloader CTF

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://learn.ChipWhisperer.io/courses/sca101). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

The goal for this lab is to attack a secure bootloader. See the accompanying `background.md` or `background.ipynb` file for background information on the bootloader. Instruction will be minimal for this lab.

You will have to:

1. Figure out how to talk to bootloader
1. Figure out what command to attack
1. Capture synchronized traces
1. Code and perform the attack

Hints are provided for each of the 4 steps. Good luck!

## Prerequisites

Hold up! Make sure that you've completed all of the previous labs before trying to tackle this one!

In [ ]:
import chipwhisperer as cw
scope = cw.scope()
target = cw.target(scope)
scope.default_setup()

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "bootloader.hex")

## General Tips

1. Commands often require a checksum, which is just the two's compliment of the 8-bit sum of the data.
2. The struct library is very useful for packing and unpacking data (i.e. turning a 16-bit number into a list of 2 bytes)
3. ChipWhisperer's serial commands (including TX) are non-blocking by default. You can check the value of `target.in_waiting()` and `target.in_waiting_tx()` to figure out when the serial message has finished transmitting (though you may want to wait a little after for the TX)

<details><summary>Part A Hint</summary>

    
```python
import time
import struct
def calc_checksum(r):
    s = 0
    for c in r:
        s += c
    cs = 0x100 - (s & 0xff)
    return cs & 0xFF

class bootloader:
    def init(self, scope, target):
        self._scope = scope
        self._target = target

    def _start_command(self, cmd):
        target = self._target
        target.flush()
        target.write([cmd]) # send mem_write command
        rtn = target.read(1).encode('latin1')[0] # get ack
        if rtn != cmd:
            raise ValueError("Invalid ack from target on command {} (got {})".format(cmd, rtn))

    def _send_password(self, password):

        # password must be 12 characters long
        password.extend([0x00] * (12 - len(password)))
        cs = calc_checksum(password)
        self._target.write(password)
        self._target.write([cs])

        time.sleep(0.1)
        rtn = self._target.read().encode('latin1')[0]

        if rtn & 0x08:
            raise ValueError("Got bad checksum response {}".format(rtn))

        if rtn & 0x01:
            return 0 # return 0 if password incorrect
        return 1 # return 1 if password correct

    def enter_bootloader(self):
        scope = self._scope
        scope.io.tio3 = 0
        time.sleep(0.1)
        scope.io.nrst = 0
        time.sleep(0.25)
        scope.io.nrst = None
        time.sleep(0.25)

    def mem_write(self, password, data):
        target = self._target
        self._start_command(0x10)

        # send password, returns 1 if correct, 0 if incorrect
        if not self._send_password(password):
            return -1

        # next send length of data
        len_data = struct.pack(">H", len(data))
        len_cs = calc_checksum(len_data)

        target.write(len_data)
        target.write([len_cs])

        rtn = self._target.read(1).encode('latin1')[0]
        if rtn & 0x08:
            raise ValueError("Got bad checksum response {}".format(rtn))

        # finally send data
        target.write(data)
        data_cs = calc_checksum(data)
        target.write(data_cs)

        rtn = target.read(1).encode('latin1')[0]
        if rtn & 0x08:
            raise ValueError("Got bad checksum response {}".format(rtn))
            
```

</details>

<details><summary>Part B Hint</summary>

To recover the password, the command you attack will need to check the password. It also can't
check the security bit, as that takes priority over the password.

The following partially completed table can be used to help you figure out which command to attack:
    
| Command | Checks Password | Security Bit |
| --- | --- | --- |
| mem_write (0x10) | Yes | Yes |
| flash_sum (0x20) | No | No |
| bootloader_info (0x30) | ??? | ??? |
| flash_erase (0x40) | ??? | ??? |
| flash_protect (0x60) | ??? | ??? |



</details>

<details><summary>Part C Hint</summary>

This target doesn't have a simple rising edge trigger for us to use, so we'll need to get creative. You may want
to revisit the lesson on the Husky's triggers to help you figure out what to use.

Note that you may need to change the value of `scope.offset` and `scope.

Note that there are multiple valid methods of triggering, including a simple rising edge trigger. Depending on what
source you choose, you may need to resynchronize your traces. You can use the following example SAD resync function to help you:

```python
import numpy as np
def sad_resync(trace, ref, ref_range):
    pattern = ref[ref_range[0]:ref_range[1]]
    off, diff = sad(trace, pattern)
    #print(off)
    actual_offset = ref_range[0] - off
    new = np.zeros(len(ref))
    try:
        if actual_offset > 0:
            new[actual_offset:] = trace[:-actual_offset]
        elif actual_offset < 0:
            new[:actual_offset] = trace[-actual_offset:]
        else:
            new[:] = trace[:]
    except:
        print(off, diff, len(new), actual_offset)
        print(new)
        print(trace)
        raise
    return new
    
def sad(trace, pattern):
    tlen = len(trace)
    plen = len(pattern)
    best_diff = 100000000
    best_offset = 0
    for i in range(tlen-plen):
        diff = np.sum(np.abs(trace[i:i+plen] - pattern[:]))
        #print(diff)
        if diff < best_diff:
            best_offset = i
            best_diff = diff
    
    return best_offset, best_diff
```

You can use it as follows:

```python
sad_resync(scope.get_last_trace(), ref_trace, ref_range)
```

</details>

<details><summary>Part D Hint</summary>

Now that we've got our attack figured out, we need to implement it now. Here's the solution from the SPA lab to get you started:

```python
password = ""
broken = False
for i in range(32):
    ref_trace = cap_pass_trace(password + "\x01\n")[0]
    biggest_diff = 0
    biggest_diff_c = ''
    for c in tqdm('abcdefghijklmnopqrstuvwxyz0123456789'): 
        trace, resp = cap_pass_trace(password + c + "\n")
        diff = np.sum(np.abs(trace - ref_trace))
        if diff > biggest_diff:
            biggest_diff = diff
            biggest_diff_c = c

        if "Access" in resp:
            broken = True
            break
            
    password += biggest_diff_c
    print(password)
    if broken:
        break

print(cap_pass_trace(password + "\n")[1])
```

</details>

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2026. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>